In [1]:
import pandas as pd

from analytics.db import query

In [2]:
query("""
      --w08
      
      SELECT 
    c.customer_state,
    ROUND(CAST(AVG(i.freight_value) AS NUMERIC), 2) AS avg_freight,
    COUNT(*) AS item_count
FROM order_items i
JOIN orders o ON i.order_id = o.order_id
JOIN customers c ON o.customer_id = c.customer_id
GROUP BY c.customer_state
ORDER BY avg_freight DESC
LIMIT 5;
      
      
      """)

,customer_state,avg_freight,item_count
0,RR,42.98,52
1,PB,42.72,602
2,RO,41.07,278
3,AC,40.07,92
4,PI,39.15,542


## Part 1 — The pipeline & Part 2 — Read the log


In [3]:
log = query("SELECT * FROM clean.cleaning_log")
log[log["rule"] == "not a city name"][["table", "key", "old_value"]]

,table,key,old_value
1434,sellers,ceb7b4fb9401cd378de7886317ad1b47,04482255
1435,sellers,1cbd32d00d01bb8087a5eb088612fd9c,sp / sp
1436,sellers,16bdc8cefd0e32a6f0824d296c5ad14a,sp
1437,sellers,2156f2671501a81034d7d07f217609d0,sp
1438,sellers,1de62b6f2fd96227629786db492433db,sp
1439,sellers,4b5f66b7adcf57f1ecc0d3c07dd6b177,vendas@creditparts.com.br
1440,sellers,0b18d63d0cd1d723567903fd34a07df2,sp


**Explanation:** This cell queries the cleaning log from the database to inspect rows flagged by the `not a city name` rule. It helps identify false positives where legitimate place names containing numbers were incorrectly treated as invalid entries.

In [4]:
log["rule"].value_counts()

rule
carrier date before approval date                    1359
city spelling                                         265
delivery date before carrier date                      23
not a city name                                         7
shipping deadline more than a year after purchase       4
a weight of 0 g is impossible                           4
no payment type and no money                            3
0 installments is impossible                            2
Name: count, dtype: int64

In [5]:
raw_count = query("SELECT COUNT(*) AS n FROM order_payments")["n"][0]
clean_count = query("SELECT COUNT(*) AS n FROM clean.order_payments")["n"][0]

raw_count, clean_count

(np.int64(103886), np.int64(103883))

## Part 3 — Checks as code


In [6]:
import pandera.pandas as pa

from analytics.clean import clean_payments
from analytics.schemas import payments_schema


In [7]:
payments = query("SELECT * FROM order_payments")
cleaned_payments, payments_log = clean_payments(payments)

try:
    payments_schema.validate(payments, lazy=True)
except pa.errors.SchemaErrors as error:
    failures = error.failure_cases

failures[["column", "check", "failure_case", "index"]]

,column,check,failure_case,index
0,payment_type,"isin(['credit_card', 'boleto', 'voucher', 'deb...",not_defined,51280
1,payment_type,"isin(['credit_card', 'boleto', 'voucher', 'deb...",not_defined,57411
2,payment_type,"isin(['credit_card', 'boleto', 'voucher', 'deb...",not_defined,94427
3,payment_installments,greater_than_or_equal_to(1),0,46982
4,payment_installments,greater_than_or_equal_to(1),0,79014


In [8]:
payments_schema.validate(cleaned_payments, lazy=True).shape

(103883, 5)